API Pipeline Notebook for Team 1000 Research Question: "Which weather factor has the biggest impact on traffic delays in New York?"

In [14]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime


We're using Open-Meteo because it has free historical weather history for a year and it says the type of weather.


In [15]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

Setup the Open-Meteo API client with cache and retry on error

In [16]:

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


Getting the different parameters we need for weather Analysis

In [17]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 40.7143,
	"longitude": -74.006,
	"start_date": "2024-01-01",
	"end_date": "2024-12-31",
	"hourly": ["temperature_2m", "snowfall", "showers", "rain", "snow_depth", "weather_code"],
	"timezone": "auto",
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch",
}
responses = openmeteo.weather_api(url, params = params)


In [18]:
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E") #New York Location
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 40.71033477783203°N -73.99308013916016°E
Elevation: 51.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s


In [65]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(1).ValuesAsNumpy()
hourly_showers = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(4).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(5).ValuesAsNumpy()

#timestamps for date and hour
timestamp = pd.date_range(
	start = pd.to_datetime(hourly.Time(),  unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)

#dictionary for the columns

hourly_data = {
    "date": timestamp.date,
    "hour": timestamp.hour
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["snowfall"] = hourly_snowfall
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["showers"] = hourly_showers

hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe['date'] = pd.to_datetime(hourly_dataframe['date']) #turn date to datetime object

print("\nHourly data\n", hourly_dataframe)

len(hourly_dataframe)
hourly_dataframe.head()


Hourly data
            date  hour  temperature_2m  snowfall      rain  weather_code  \
0    2024-01-01     4       38.763500       0.0  0.000000           3.0   
1    2024-01-01     5       40.923500       0.0  0.000000           3.0   
2    2024-01-01     6       40.653500       0.0  0.000000           3.0   
3    2024-01-01     7       40.653500       0.0  0.000000           3.0   
4    2024-01-01     8       39.663502       0.0  0.000000           0.0   
...         ...   ...             ...       ...       ...           ...   
8779 2024-12-31    23       46.683498       0.0  0.000000           3.0   
8780 2025-01-01     0       47.403503       0.0  0.000000           3.0   
8781 2025-01-01     1       47.313499       0.0  0.000000           3.0   
8782 2025-01-01     2       47.043499       0.0  0.015748          51.0   
8783 2025-01-01     3       43.983498       0.0  0.015748          51.0   

      snow_depth  showers  
0            0.0      0.0  
1            0.0      0.0  
2

,date,hour,temperature_2m,snowfall,rain,weather_code,snow_depth,showers
0,2024-01-01,4,38.763500,0.0,0.0,3.0,0.0,0.0
1,2024-01-01,5,40.923500,0.0,0.0,3.0,0.0,0.0
2,2024-01-01,6,40.653500,0.0,0.0,3.0,0.0,0.0
3,2024-01-01,7,40.653500,0.0,0.0,3.0,0.0,0.0
4,2024-01-01,8,39.663502,0.0,0.0,0.0,0.0,0.0


Now we will obtain traffic Data from NYC Open Data.  I will be using the Automated Traffic Volume Counts from DOT excel, it records volume of cars in different streets in boroughs at a certain hour. Before I upload the csv I filter the traffic report to been the year 2024 and the time to be exactly at the top of the hour. 

In [ ]:
df = pd.read_csv('Automated_Traffic_Volume_Counts.csv')

#make month, date, year column into one column

df.columns

df["date"] = pd.to_datetime({
    'year': df['Yr'],
    'month': df['M'],
    'day': df['D']
    })

df = df.drop(['M', 'D', 'Yr', 'RequestID', 'SegmentID'], axis=1)

df.head()



23802

Now we must connect the Traffic and Weather Data that have the matching date and Month way we can observe trends

In [86]:
weather_traffic_df = pd.merge(
    hourly_dataframe, 
    df, 
    left_on=['date', 'hour'], 
    right_on=['date', 'HH'], 
    how='inner'
)

weather_traffic_df.columns

weather_traffic_df = weather_traffic_df.drop(['HH'], axis=1)

weather_traffic_df.head()

#weather_traffic_df.to_csv('Weather_Traffic.csv')

,date,hour,temperature_2m,snowfall,rain,weather_code,snow_depth,showers,Boro,Vol,street,fromSt,toSt,Direction
0,2024-01-06,0,32.733501,0.0,0.0,0.0,0.0,0.0,Queens,61,BROADWAY,Queens Boulevard,Queens Boulevard Line,SB
1,2024-01-06,0,32.733501,0.0,0.0,0.0,0.0,0.0,Queens,5,GRAND AVENUE,72 Street,Mazeau Street,EB
2,2024-01-06,0,32.733501,0.0,0.0,0.0,0.0,0.0,Queens,32,BROADWAY,Queens Boulevard Line,Queens Boulevard,NB
3,2024-01-06,0,32.733501,0.0,0.0,0.0,0.0,0.0,Brooklyn,9,METROPOLITAN AVENUE,Dead end,Varick Avenue,WB
4,2024-01-06,0,32.733501,0.0,0.0,0.0,0.0,0.0,Manhattan,5,9 AVENUE,West 219 Street,West 220 Street,SB
